# LoggingMiddleware

`before_agent` 把计时起点写入本次运行的私有状态，`after_agent` 在 Agent 正常结束时记录总耗时（秒）。`wrap_tool_call` 在执行工具前记录工具名称。同步和异步调用使用相同的日志格式。

In [1]:
import logging
from collections.abc import Awaitable, Callable
from time import perf_counter
from typing import Annotated, NotRequired

from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.agents.middleware.types import PrivateStateAttr, ToolCallRequest
from langchain.messages import ToolMessage
from langgraph.runtime import Runtime
from langgraph.types import Command

logger = logging.getLogger("agent.middleware")


class LoggingState(AgentState):
    logging_started_at: NotRequired[Annotated[float, PrivateStateAttr]]


class LoggingMiddleware(AgentMiddleware[LoggingState]):
    state_schema = LoggingState

    def before_agent(self, state: LoggingState, runtime: Runtime) -> dict[str, float]:
        return {"logging_started_at": perf_counter()}

    def after_agent(self, state: LoggingState, runtime: Runtime) -> None:
        started_at = state.get("logging_started_at")
        if started_at is not None:
            logger.info("Agent 运行耗时 %.3f 秒", perf_counter() - started_at)

    async def abefore_agent(
        self, state: LoggingState, runtime: Runtime
    ) -> dict[str, float]:
        return self.before_agent(state, runtime)

    async def aafter_agent(self, state: LoggingState, runtime: Runtime) -> None:
        self.after_agent(state, runtime)

    def wrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], ToolMessage | Command],
    ) -> ToolMessage | Command:
        logger.info("开始调用工具 %s", request.tool_call["name"])
        return handler(request)

    async def awrap_tool_call(
        self,
        request: ToolCallRequest,
        handler: Callable[[ToolCallRequest], Awaitable[ToolMessage | Command]],
    ) -> ToolMessage | Command:
        logger.info("开始调用工具 %s", request.tool_call["name"])
        return await handler(request)

## 运行示例

模拟模型先请求调用 `add`，再返回最终回答。示例会实际执行工具，并依次输出工具名称和 Agent 总耗时，无需 API 密钥。

In [2]:
from langchain.agents import create_agent
from langchain.messages import AIMessage
from langchain.tools import tool
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel


class DemoChatModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        return self  # 工具调用已在 responses 中预设。


@tool
def add(a: int, b: int) -> int:
    """计算两个整数的和。"""
    return a + b


logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger.setLevel(logging.INFO)

model = DemoChatModel(
    responses=[
        AIMessage(
            content="",
            tool_calls=[{"name": "add", "args": {"a": 2, "b": 3}, "id": "call_add"}],
        ),
        AIMessage(content="2 + 3 = 5"),
    ]
)
agent = create_agent(model=model, tools=[add], middleware=[LoggingMiddleware()])
result = agent.invoke({"messages": [{"role": "user", "content": "2 加 3 等于多少？"}]})
print(result["messages"][-1].content)

INFO 开始调用工具 add
INFO Agent 运行耗时 0.010 秒


2 + 3 = 5
